# Apriori Algorithm Implementation Assignment

### Objective:
You will implement the **Apriori algorithm** from scratch (i.e., without using any libraries like `mlxtend`) to find frequent itemsets and generate association rules.

### Dataset:
Use the [Online Retail Dataset](https://www.kaggle.com/datasets/vijayuv/onlineretail) from Kaggle. You can filter it for a specific country (e.g., `United Kingdom`) and time range to reduce size if needed.

---

## Step 1: Data Preprocessing

- Load the dataset
- Remove rows with missing values
- Filter out rows where `Quantity <= 0`
- Convert Data into Basket Format

👉 **Implement code below**

In [2]:
import pandas as pd

# Load dataset
df = pd.read_csv('OnlineRetail.csv', encoding='ISO-8859-1')   # update with actual path

# Preprocessing
df = df.dropna()
df = df[df['Quantity'] > 0]
df = df[df['Country'] == 'United Kingdom']

# Basket format
basket = df.groupby(['InvoiceNo', 'Description'])['Quantity'].sum().unstack().reset_index().fillna(0).set_index('InvoiceNo')
basket = basket.applymap(lambda x: 1 if x > 0 else 0)

transactions = basket.values.tolist()
items = basket.columns.tolist()

C:\Users\Pooja\AppData\Local\Temp\ipykernel_12960\4096613161.py:13: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  basket = basket.applymap(lambda x: 1 if x > 0 else 0)


## Step 2: Implement Apriori Algorithm
Step-by-Step Procedure...

In [3]:
from itertools import combinations

def calculate_support(transactions, candidates, min_support):
    item_count = {}
    total = len(transactions)

    for transaction in transactions:
        tset = set([items[i] for i, val in enumerate(transaction) if val == 1])
        for candidate in candidates:
            if candidate.issubset(tset):
                item_count[candidate] = item_count.get(candidate, 0) + 1

    # Convert to support
    return {item: count/total for item, count in item_count.items() if (count/total) >= min_support}

def generate_candidates(prev_frequent_itemsets, k):
    candidates = set()
    prev_items = list(prev_frequent_itemsets.keys())
    for i in range(len(prev_items)):
        for j in range(i+1, len(prev_items)):
            union = prev_items[i].union(prev_items[j])
            if len(union) == k:
                candidates.add(union)
    return candidates

def get_frequent_itemsets(transactions, min_support):
    # Step 1: Frequent 1-itemsets
    single_items = [{item} for item in items]
    frequent_itemsets = []

    L1 = calculate_support(transactions, single_items, min_support)
    current_L = L1
    k = 2

    while current_L:
        frequent_itemsets.append(current_L)
        candidates = generate_candidates(current_L, k)
        current_L = calculate_support(transactions, candidates, min_support)
        k += 1

    return frequent_itemsets

## Step 3: Generate Association Rules
- Support, Confidence calculation

In [4]:
def generate_rules(frequent_itemsets, min_confidence):
    rules = []
    support_data = {}

    for level in frequent_itemsets:
        support_data.update(level)

    for itemset in support_data.keys():
        if len(itemset) > 1:
            for i in range(1, len(itemset)):
                for antecedent in combinations(itemset, i):
                    antecedent = frozenset(antecedent)
                    consequent = itemset - antecedent
                    if support_data[antecedent] > 0:
                        confidence = support_data[itemset] / support_data[antecedent]
                        if confidence >= min_confidence:
                            rules.append({
                                'antecedent': antecedent,
                                'consequent': consequent,
                                'support': support_data[itemset],
                                'confidence': confidence
                            })
    return rules

## Step 4: Output and Visualize
- Print top 10 frequent itemsets
- Print top 10 rules

In [8]:
def generate_rules(frequent_itemsets, min_confidence):
    rules = []
    support_data = {}

    # Collect support values for all itemsets
    for level in frequent_itemsets:
        support_data.update(level)

    # Generate rules
    for itemset in support_data.keys():
        if len(itemset) > 1:
            for i in range(1, len(itemset)):
                for antecedent in combinations(itemset, i):
                    antecedent = frozenset(antecedent)
                    consequent = itemset - antecedent
                    if support_data[antecedent] > 0:
                        confidence = support_data[itemset] / support_data[antecedent]
                        if confidence >= min_confidence:
                            rules.append({
                                'antecedent': antecedent,
                                'consequent': consequent,
                                'support': support_data[itemset],
                                'confidence': confidence
                            })
    return rules


# -----------------------
# Run Apriori and print
# -----------------------

min_support = 0.02   # 2%
min_confidence = 0.5 # 50%

frequent_itemsets = get_frequent_itemsets(transactions, min_support)
rules = generate_rules(frequent_itemsets, min_confidence)

# Collect all frequent itemsets in one list
all_freq = []
for level in frequent_itemsets:
    all_freq.extend([(list(k), v) for k, v in level.items()])

# Sort by support
all_freq = sorted(all_freq, key=lambda x: x[1], reverse=True)

print("🔹 Top 10 Frequent Itemsets:")
for itemset, support in all_freq[:10]:
    print(itemset, "=> Support:", round(support, 3))

# Sort rules by confidence
rules = sorted(rules, key=lambda x: x['confidence'], reverse=True)

print("\n🔹 Top 10 Association Rules:")
for r in rules[:10]:
    print(list(r['antecedent']), "=>", list(r['consequent']),
          "| Support:", round(r['support'], 3),
          "| Confidence:", round(r['confidence'], 3))


🔹 Top 10 Frequent Itemsets:
['WHITE HANGING HEART T-LIGHT HOLDER'] => Support: 0.113
['JUMBO BAG RED RETROSPOT'] => Support: 0.087
['REGENCY CAKESTAND 3 TIER'] => Support: 0.085
['ASSORTED COLOUR BIRD ORNAMENT'] => Support: 0.078
['PARTY BUNTING'] => Support: 0.078
['LUNCH BAG RED RETROSPOT'] => Support: 0.067
['SET OF 3 CAKE TINS PANTRY DESIGN '] => Support: 0.06
['LUNCH BAG  BLACK SKULL.'] => Support: 0.06
["PAPER CHAIN KIT 50'S CHRISTMAS "] => Support: 0.057
['NATURAL SLATE HEART CHALKBOARD '] => Support: 0.056

🔹 Top 10 Association Rules:
['PINK REGENCY TEACUP AND SAUCER', 'ROSES REGENCY TEACUP AND SAUCER '] => ['GREEN REGENCY TEACUP AND SAUCER'] | Support: 0.02 | Confidence: 0.89
['GREEN REGENCY TEACUP AND SAUCER', 'PINK REGENCY TEACUP AND SAUCER'] => ['ROSES REGENCY TEACUP AND SAUCER '] | Support: 0.02 | Confidence: 0.844
['PINK REGENCY TEACUP AND SAUCER'] => ['GREEN REGENCY TEACUP AND SAUCER'] | Support: 0.024 | Confidence: 0.819
['GREEN REGENCY TEACUP AND SAUCER'] => ['ROSES RE